# Обучение ResNet18-регрессора позы

Этот notebook обучает CV-часть проекта: ResNet18-регрессор, который предсказывает позу аппарата по сгенерированным кадрам LunarLander.

## Датасет и определение модели

`LunarLanderVisionDataset` читает `metadata.json` выбранной CV-интеграции, берёт общие `images_dir` и `labels_file`, а затем формирует target только из колонок, указанных в `target_columns`. Для `theta` используется представление `[sin(theta), cos(theta)]`, чтобы убрать разрыв около `-pi` и `pi`.

In [ ]:
import csv
import json
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset


class LunarLanderVisionDataset(Dataset):
    def __init__(
        self,
        metadata_path: str,
        augment: bool = False,
        particle_prob: float = 0.35,
        seed: int = 42,
    ):
        self.metadata_path = Path(metadata_path)
        self.metadata = json.loads(self.metadata_path.read_text(encoding="utf-8"))
        self.integration_dir = self.metadata_path.parent
        self.images_dir = (self.integration_dir / self.metadata["images_dir"]).resolve()
        self.labels_file = (self.integration_dir / self.metadata["labels_file"]).resolve()
        self.target_columns = list(self.metadata["target_columns"])
        self.output_columns = self._make_output_columns(self.target_columns)
        self.samples = []
        self.augment = augment
        self.particle_prob = particle_prob
        self.rng = np.random.default_rng(seed)

        with open(self.labels_file, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            missing_columns = {"image_name", *self.target_columns} - set(reader.fieldnames or [])
            if missing_columns:
                raise ValueError(f"В labels.csv нет колонок: {sorted(missing_columns)}")

            for row in reader:
                sample = {"image_name": row["image_name"]}
                for key, value in row.items():
                    if key == "image_name" or value in (None, ""):
                        continue
                    sample[key] = float(value)
                self.samples.append(sample)

    @staticmethod
    def _make_output_columns(target_columns):
        output_columns = []
        for column in target_columns:
            if column == "theta":
                output_columns.extend(["sin_theta", "cos_theta"])
            else:
                output_columns.append(column)
        return output_columns

    def _draw_disk(self, image, cx, cy, radius, color, alpha=0.6):
        h, w, _ = image.shape

        x_min = max(0, cx - radius)
        x_max = min(w, cx + radius + 1)
        y_min = max(0, cy - radius)
        y_max = min(h, cy + radius + 1)

        if x_min >= x_max or y_min >= y_max:
            return image

        yy, xx = np.ogrid[y_min:y_max, x_min:x_max]
        mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= radius ** 2

        patch = image[y_min:y_max, x_min:x_max].copy()
        patch[mask] = (1.0 - alpha) * patch[mask] + alpha * color
        image[y_min:y_max, x_min:x_max] = patch

        return image

    def __len__(self):
        return len(self.samples)

    def _obs_to_pixel(self, x_obs, y_obs, h, w):
        px = int(np.clip((x_obs + 1.0) * 0.5 * w, 0, w - 1))
        py = int(np.clip(h * (0.705 - 0.5 * y_obs), 0, h - 1))
        return px, py

    def _add_engine_particles_realistic(self, image, x_obs, y_obs, theta):
        if self.rng.random() > self.particle_prob:
            return image

        h, w, _ = image.shape
        cx, cy = self._obs_to_pixel(x_obs, y_obs, h, w)

        body_size = max(10.0, 0.03 * w)

        down = np.array([np.sin(theta), np.cos(theta)], dtype=np.float32)
        right = np.array([np.cos(theta), -np.sin(theta)], dtype=np.float32)

        center = np.array([cx, cy], dtype=np.float32)
        main_nozzle = center + 0.85 * body_size * down
        left_nozzle = center + 0.65 * body_size * down - 0.50 * body_size * right
        right_nozzle = center + 0.65 * body_size * down + 0.50 * body_size * right

        use_main = self.rng.random() < 0.75
        use_left = self.rng.random() < 0.30
        use_right = self.rng.random() < 0.30

        engine_specs = []
        if use_main:
            engine_specs.append((main_nozzle, 4, 8, 0.25, 1.30))
        if use_left:
            engine_specs.append((left_nozzle, 2, 4, 0.15, 0.75))
        if use_right:
            engine_specs.append((right_nozzle, 2, 4, 0.15, 0.75))

        for nozzle, n_min, n_max, spread_scale, length_scale in engine_specs:
            n_particles = int(self.rng.integers(n_min, n_max + 1))

            for _ in range(n_particles):
                dist = float(self.rng.uniform(0.25 * body_size, length_scale * body_size))
                lateral = float(self.rng.normal(0.0, spread_scale * body_size))

                pos = nozzle + dist * down + lateral * right

                px = int(round(pos[0]))
                py = int(round(pos[1]))

                radius = int(self.rng.integers(2, 6))
                alpha = float(self.rng.uniform(0.35, 0.75))

                color = np.array([
                    self.rng.uniform(0.90, 1.00),
                    self.rng.uniform(0.10, 0.35),
                    self.rng.uniform(0.00, 0.08),
                ], dtype=np.float32)

                image = self._draw_disk(image, px, py, radius, color, alpha=alpha)

                if self.rng.random() < 0.35:
                    core_radius = max(1, radius - 1)
                    core_color = np.array([1.0, 0.85, 0.15], dtype=np.float32)
                    image = self._draw_disk(
                        image,
                        px,
                        py,
                        core_radius,
                        core_color,
                        alpha=min(0.9, alpha + 0.15),
                    )

        return image

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = np.load(self.images_dir / sample["image_name"])

        image = image.astype(np.float32) / 255.0

        if self.augment and all(key in sample for key in ("x", "y", "theta")):
            image = self._add_engine_particles_realistic(
                image,
                sample["x"],
                sample["y"],
                sample["theta"],
            )

        image = np.transpose(image[:, :, :3], (2, 0, 1))

        target_values = []
        for column in self.target_columns:
            value = float(sample[column])
            if column == "theta":
                target_values.extend([np.sin(value), np.cos(value)])
            else:
                target_values.append(value)

        target = np.array(target_values, dtype=np.float32)
        return torch.tensor(image), torch.tensor(target)


import torch
import torch.nn as nn
from torchvision.models import resnet18


class StateRegressorResNet18(nn.Module):
    def __init__(self, out_dim: int = 4):
        super().__init__()

        self.backbone = resnet18(weights=None)
        in_features = self.backbone.fc.in_features

        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(p=0.1),
            nn.Linear(128, out_dim)
        )

    def forward(self, x):
        return self.backbone(x)

## Цикл обучения

Функция обучения делает train/validation-разбиение 80/20, применяет лёгкую аугментацию частиц двигателя и оптимизирует ResNet18-регрессор по среднеквадратичной ошибке.

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from torch.utils.data import DataLoader, Subset

def train_state_regressor(
    dataset_dir: str,
    batch_size: int = 32,
    lr: float = 1e-5,
    num_epochs: int = 20,
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
):
    dataset = LunarLanderVisionDataset(dataset_dir, augment=True)
    n_total = len(dataset)
    n_train = int(0.8 * n_total)
    n_val = n_total - n_train
    train_ds, val_ds = random_split(dataset, [n_train, n_val])
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    model = StateRegressorResNet18(out_dim=len(dataset.output_columns)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.MSELoss()

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        for images, targets in train_loader:
            images = images.to(device)
            targets = targets.to(device)

            preds = model(images)
            loss = criterion(preds, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for images, targets in val_loader:
                images = images.to(device)
                targets = targets.to(device)

                preds = model(images)
                loss = criterion(preds, targets)
                val_loss += loss.item() * images.size(0)

        val_loss /= len(val_loader.dataset)

        print(
            f"Эпоха {epoch+1:02d} | "
            f"ошибка_обучения={train_loss:.6f} | ошибка_валидации={val_loss:.6f}"
        )

    return model

## Запуск обучения

Запустите эту ячейку, чтобы обучить новый CV-checkpoint на конфиге `../../data/cv_integrations/x_y_theta/metadata.json`. Если нужен другой эксперимент, измените путь к `metadata.json`, `batch_size`, `lr` или `num_epochs`.

In [ ]:
model = train_state_regressor("../../data/cv_integrations/x_y_theta/metadata.json")

## Оценка ошибки позы

Оценим обученную модель на случайном подмножестве сохранённых кадров. Вывод покажет MAE для `x`, `y` и циклически нормализованную угловую ошибку для `theta`.

In [ ]:
import torch
from torch.utils.data import DataLoader

dataset = LunarLanderVisionDataset("../../data/cv_integrations/x_y_theta/metadata.json")

rng = np.random.default_rng(42)
indices = rng.choice(len(dataset), size=10000, replace=False)

small_dataset = Subset(dataset, indices)
device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)
model.eval()

loader = DataLoader(small_dataset, batch_size=32, shuffle=False)

model.eval()
mae_xy_sum = torch.zeros(2, device=device)
theta_mae_sum = 0.0
num_samples = 0

with torch.no_grad():
    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        preds = model(images)

        # x, y
        mae_xy_sum += (preds[:, :2] - targets[:, :2]).abs().sum(dim=0)

        # угол
        pred_vec = preds[:, 2:4]
        pred_vec = pred_vec / (pred_vec.norm(dim=1, keepdim=True) + 1e-8)

        theta_pred = torch.atan2(pred_vec[:, 0], pred_vec[:, 1])
        theta_true = torch.atan2(targets[:, 2], targets[:, 3])

        theta_err = torch.atan2(
            torch.sin(theta_pred - theta_true),
            torch.cos(theta_pred - theta_true)
        ).abs()

        theta_mae_sum += theta_err.sum().item()
        num_samples += images.size(0)

mae_xy = mae_xy_sum / num_samples
theta_mae = theta_mae_sum / num_samples

print("MAE по x, y:", mae_xy.cpu().numpy())
print("MAE по theta:", theta_mae)

## Сохранение CV-checkpoint

Сохраняем обученные веса в раздел CV-checkpoint'ов, который используют RL-скрипты и примеры инференса.

In [ ]:
torch.save(model.state_dict(), "../../checkpoints/cv/resnet18_pose/state_regressor_resnet18.pth")